In [1]:
# %%
# =============================================================================
# 10_rag_query_rewriting.ipynb
# Financial AI Governance — RAG + Query Rewriting (Option B)
# Kernel : Python (llm_env)
# Input  : data/processed/dataset_final.json
#          vectordb/ (Chroma persistent stores from 02_rag_pipeline.ipynb)
# Output : results/responses/responses_rag_rewrite.json
#          results/tables/table_rewrite_summary.csv
# Note   : Original question is rewritten by LLM before retrieval (top-k=3).
#          Inference prompt is identical to 03_llm_inference.ipynb (RAG).
#          Only the retrieval query differs from the standard RAG condition.
# =============================================================================

# %%
# =============================================================================
# Cell 1. Libraries and Environment Setup
# =============================================================================
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Directory paths — identical to 03_llm_inference.ipynb
DATA_DIR     = '../data/processed'
VDB_DIR      = '../vectordb'
RESPONSE_DIR = '../results/responses'
TABLE_DIR    = '../results/tables'

for d in [RESPONSE_DIR, TABLE_DIR]:
    os.makedirs(d, exist_ok=True)

# API setup
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
LLM_MODEL      = os.getenv('LLM_MODEL', 'gpt-4o-mini')
EMBED_MODEL    = 'text-embedding-3-small'

if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env")

client     = OpenAI(api_key=OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(model=EMBED_MODEL, api_key=OPENAI_API_KEY)

TOP_K = 3  # same as standard RAG — only query changes

print(f"[INFO] LLM model      : {LLM_MODEL}")
print(f"[INFO] Embedding model: {EMBED_MODEL}")
print(f"[INFO] Top-k          : {TOP_K} (fixed — query rewriting only)")


# %%
# =============================================================================
# Cell 2. Load Dataset and Vector Stores
# =============================================================================
with open(os.path.join(DATA_DIR, 'dataset_final.json'), 'r', encoding='utf-8') as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)
print(f"[INFO] Dataset loaded: {len(df)} records")

# Load Chroma vector stores — identical to 03_llm_inference.ipynb
VDB_CONFIG = {
    'NIST_AI_RMF'    : {'persist_dir': os.path.join(VDB_DIR, 'nist'),          'collection': 'nist_ai_rmf'},
    'KR_AI_BASIC_ACT': {'persist_dir': os.path.join(VDB_DIR, 'kr_aibasicact'), 'collection': 'kr_aibasicact'},
    'EU_AI_ACT'      : {'persist_dir': os.path.join(VDB_DIR, 'eu_aiact'),      'collection': 'eu_aiact'},
}

vector_stores = {}
for reg_key, cfg in VDB_CONFIG.items():
    vector_stores[reg_key] = Chroma(
        collection_name    = cfg['collection'],
        embedding_function = embeddings,
        persist_directory  = cfg['persist_dir'],
    )
    count = vector_stores[reg_key]._collection.count()
    print(f"  [LOAD] {reg_key:20s} | {count} chunks")

print("[INFO] All vector stores loaded.")


# %%
# =============================================================================
# Cell 3. Prompt Templates
# =============================================================================
# --- Query Rewriting Prompt ---
# Expands abbreviations, adds regulatory context, improves retrieval specificity.
# Grounded in chan2024rqrag (RQ-RAG) methodology.

REG_NAME_MAP = {
    'NIST_AI_RMF'    : 'NIST AI Risk Management Framework (AI RMF 1.0)',
    'KR_AI_BASIC_ACT': 'Korean AI Basic Act (Law No. 21311)',
    'EU_AI_ACT'      : 'EU AI Act (Regulation 2024/1689)',
}

REWRITE_SYSTEM_PROMPT = """You are an AI governance expert specializing in financial institution 
regulatory compliance. Your task is to rewrite questions to improve their retrieval effectiveness 
against regulatory documents."""

REWRITE_USER_TEMPLATE = """Rewrite the following question to improve retrieval from regulatory documents.

Guidelines:
- Expand abbreviations (e.g., "AI" → "artificial intelligence", "RMS" → "risk management system")
- Add relevant regulatory terminology specific to {regulation_name}
- Make implicit scope explicit (e.g., "high-risk" → "High-Risk AI systems under Annex III")
- Keep the core meaning unchanged
- Output only the rewritten question, no explanation

Original question: {question}
Regulation: {regulation_name}

Rewritten question:"""

# --- Inference Prompts (identical to 03_llm_inference.ipynb) ---
SYSTEM_PROMPT = """You are an expert AI governance advisor specializing in financial institution AI compliance.
Your role is to support an AI Review Committee at a financial institution by providing accurate,
regulation-grounded answers to governance questions.

When answering:
1. Cite specific regulatory provisions (article numbers, section codes) where applicable.
2. Identify the governance axis: G1 (Accuracy), G2 (Safety), G3 (Transparency), or G4 (Compliance).
3. Flag high-risk scenarios and recommend human oversight where appropriate.
4. If uncertain, state limitations clearly rather than fabricating information.
5. Keep answers concise, structured, and actionable for a compliance committee."""


def build_rag_prompt(question: str, context: str) -> str:
    """Identical to 03_llm_inference.ipynb build_rag_prompt()."""
    return f"""Answer the following AI governance question using the regulatory context provided below.

--- REGULATORY CONTEXT ---
{context}
--- END CONTEXT ---

Question: {question}

Provide a structured answer grounded in the regulatory context above.
Cite specific article numbers or section codes from the context where applicable."""


# %%
# =============================================================================
# Cell 4. Query Rewriting and Retrieval Functions
# =============================================================================
def rewrite_query(question: str, regulation: str) -> str:
    """
    Rewrite a question using LLM to improve retrieval specificity.
    Based on RQ-RAG methodology (Chan et al., 2024).

    Args:
        question   : Original question string
        regulation : Regulation key (e.g., 'NIST_AI_RMF')

    Returns:
        Rewritten question string
    """
    reg_name = REG_NAME_MAP.get(regulation, regulation)
    user_prompt = REWRITE_USER_TEMPLATE.format(
        question        = question,
        regulation_name = reg_name,
    )
    try:
        res = client.chat.completions.create(
            model       = LLM_MODEL,
            temperature = 0.0,
            max_tokens  = 200,
            messages    = [
                {'role': 'system', 'content': REWRITE_SYSTEM_PROMPT},
                {'role': 'user',   'content': user_prompt},
            ]
        )
        rewritten = res.choices[0].message.content.strip()
        tokens    = res.usage.total_tokens
        return rewritten, tokens
    except Exception as e:
        # Fallback: return original question on error
        print(f"  [WARN] Rewrite failed — using original. Error: {e}")
        return question, 0


def retrieve_context(question: str, regulation: str, k: int = TOP_K) -> str:
    """
    Retrieve top-k chunks using the (rewritten) question.
    Identical logic to 03_llm_inference.ipynb retrieve_context().
    """
    if regulation not in vector_stores:
        raise ValueError(f"[ERROR] Unknown regulation: {regulation}")
    docs    = vector_stores[regulation].similarity_search(question, k=k)
    context = "\n\n---\n\n".join([d.page_content for d in docs])
    return context


def call_llm(system_prompt: str, user_prompt: str,
             model: str = LLM_MODEL,
             temperature: float = 0.0,
             max_tokens: int = 1000) -> dict:
    """Identical to 03_llm_inference.ipynb call_llm()."""
    try:
        res = client.chat.completions.create(
            model       = model,
            temperature = temperature,
            max_tokens  = max_tokens,
            messages    = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_prompt},
            ]
        )
        return {
            'response'         : res.choices[0].message.content.strip(),
            'prompt_tokens'    : res.usage.prompt_tokens,
            'completion_tokens': res.usage.completion_tokens,
            'total_tokens'     : res.usage.total_tokens,
        }
    except Exception as e:
        return {
            'response'         : f'[ERROR] {str(e)}',
            'prompt_tokens'    : 0,
            'completion_tokens': 0,
            'total_tokens'     : 0,
        }


# %%
# =============================================================================
# Cell 5. Query Rewriting Spot-Check
# =============================================================================
# Verify rewriting quality before full run

TEST_QUERIES = [
    {'question'  : 'What are the requirements for human oversight of '
                   'high-risk AI credit scoring systems?',
     'regulation': 'EU_AI_ACT'},
    {'question'  : 'What obligations apply to AI business operators '
                   'providing High-Impact AI for credit screening?',
     'regulation': 'KR_AI_BASIC_ACT'},
    {'question'  : 'How does the GOVERN function address legal and '
                   'regulatory requirements for AI systems?',
     'regulation': 'NIST_AI_RMF'},
]

print("[INFO] Query Rewriting Spot-Check\n")
print("=" * 70)

for t in TEST_QUERIES:
    rewritten, tok = rewrite_query(t['question'], t['regulation'])
    print(f"Regulation : {t['regulation']}")
    print(f"Original   : {t['question']}")
    print(f"Rewritten  : {rewritten}")
    print(f"Tokens used: {tok}")
    print("-" * 70)
    time.sleep(0.3)


# %%
# =============================================================================
# Cell 6. Run RAG + Query Rewriting Inference
# =============================================================================
print(f"[RUN] RAG + Query Rewriting inference")
print(f"      Model: {LLM_MODEL} | Temperature: 0.0 | "
      f"Max tokens: 1000 | Top-k: {TOP_K}")
print(f"      Total records: {len(df)}\n")

results           = []
total_tokens_inf  = 0   # inference tokens
total_tokens_rew  = 0   # rewriting tokens

for _, row in tqdm(df.iterrows(), total=len(df), desc='RAG+Rewrite'):
    # Step 1: Rewrite query
    rewritten_q, rew_tok = rewrite_query(row['question'], row['regulation'])
    total_tokens_rew += rew_tok
    time.sleep(0.1)

    # Step 2: Retrieve using rewritten query
    context = retrieve_context(rewritten_q, row['regulation'], k=TOP_K)

    # Step 3: Inference using ORIGINAL question (not rewritten)
    # Rewriting is only for retrieval — inference sees the original question
    user_prompt = build_rag_prompt(row['question'], context)
    result      = call_llm(SYSTEM_PROMPT, user_prompt)

    # Field structure identical to 03_llm_inference.ipynb
    results.append({
        'id'                  : row['id'],
        'scenario_id'         : row['scenario_id'],
        'regulation'          : row['regulation'],
        'function'            : row['function'],
        'difficulty'          : row['difficulty'],
        'financial_domain'    : row['financial_domain'],
        'risk_level'          : row['risk_level'],
        'governance_axis'     : row['governance_axis'],
        'question'            : row['question'],
        'ground_truth'        : row['ground_truth'],
        'legal_basis'         : row['legal_basis'],
        'condition'           : 'rag_rewrite',
        'top_k'               : TOP_K,
        'rewritten_question'  : rewritten_q,   # additional field
        'rewrite_tokens'      : rew_tok,        # additional field
        'context_used'        : context,
        'response'            : result['response'],
        'prompt_tokens'       : result['prompt_tokens'],
        'completion_tokens'   : result['completion_tokens'],
        'total_tokens'        : result['total_tokens'],
        'model'               : LLM_MODEL,
        'temperature'         : 0.0,
    })

    total_tokens_inf += result['total_tokens']
    time.sleep(0.2)

    idx = len(results)
    if idx % 50 == 0:
        errors  = sum(1 for r in results if r['response'].startswith('[ERROR]'))
        avg_len = sum(len(r['response']) for r in results) / idx
        print(f"  [Checkpoint {idx:3d}/300] errors: {errors} | "
              f"avg response: {avg_len:.0f} chars | "
              f"inf tokens: {total_tokens_inf:,} | "
              f"rew tokens: {total_tokens_rew:,}")

# Save
out_path = os.path.join(RESPONSE_DIR, 'responses_rag_rewrite.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

errors = sum(1 for r in results if r['response'].startswith('[ERROR]'))
total_tokens_all = total_tokens_inf + total_tokens_rew
print(f"\n[SAVE] responses_rag_rewrite.json")
print(f"[INFO] records: {len(results)} | errors: {errors}")
print(f"[INFO] inference tokens : {total_tokens_inf:,}")
print(f"[INFO] rewrite tokens   : {total_tokens_rew:,}")
print(f"[INFO] total tokens     : {total_tokens_all:,} | "
      f"estimated cost: ${total_tokens_all * 0.00000015:.4f}")


# %%
# =============================================================================
# Cell 7. Rewriting Quality Analysis
# =============================================================================
# Compare original vs rewritten questions to verify meaningful transformation

df_results = pd.DataFrame(results)

print("[INFO] Rewriting Quality Analysis\n")

# Length comparison
orig_len = df_results['question'].str.len().mean()
rew_len  = df_results['rewritten_question'].str.len().mean()
print(f"  Avg original question length  : {orig_len:.0f} chars")
print(f"  Avg rewritten question length : {rew_len:.0f} chars")
print(f"  Avg length increase           : {rew_len - orig_len:+.0f} chars "
      f"({(rew_len/orig_len - 1)*100:+.1f}%)")

# Rewrite token usage by regulation
print(f"\n  Rewrite token usage by regulation:")
rew_tok_by_reg = df_results.groupby('regulation')['rewrite_tokens'].agg(['mean','sum'])
for reg, row_ in rew_tok_by_reg.iterrows():
    print(f"    {reg:20s} | avg: {row_['mean']:.0f} | total: {int(row_['sum']):,}")

# Sample rewrite pairs (3 examples)
print(f"\n  Sample rewrite pairs:")
print("=" * 70)
for _, r in df_results.sample(3, random_state=42).iterrows():
    print(f"  Regulation : {r['regulation']}")
    print(f"  Original   : {r['question'][:100]}...")
    print(f"  Rewritten  : {r['rewritten_question'][:100]}...")
    print("-" * 70)


# %%
# =============================================================================
# Cell 8. Summary Comparison — RAG vs RAG+Rewrite
# =============================================================================
# Load standard RAG k=3 for direct comparison

rag_k3_path = os.path.join(RESPONSE_DIR, 'responses_rag.json')
with open(rag_k3_path, 'r', encoding='utf-8') as f:
    rag_k3_data = json.load(f)
print(f"[LOAD] responses_rag.json: {len(rag_k3_data)} records")

comparison = {
    'rag_k3'    : rag_k3_data,
    'rag_rewrite': results,
}

rows = []
for label, data in comparison.items():
    avg_ctx   = sum(len(r.get('context_used', '')) for r in data) / len(data)
    avg_resp  = sum(len(r.get('response', ''))      for r in data) / len(data)
    avg_tok   = sum(r.get('total_tokens', 0)        for r in data) / len(data)
    total_tok = sum(r.get('total_tokens', 0)        for r in data)
    errors    = sum(1 for r in data if r.get('response','').startswith('[ERROR]'))
    rows.append({
        'Condition'           : label,
        'N'                   : len(data),
        'Top-k'               : TOP_K,
        'Avg Context (chars)' : round(avg_ctx),
        'Avg Response (chars)': round(avg_resp),
        'Avg Total Tokens'    : round(avg_tok),
        'Total Tokens'        : total_tok,
        'Errors'              : errors,
        'Est. Cost ($)'       : round(total_tok * 0.00000015, 4),
    })

df_summary = pd.DataFrame(rows)
print("\n[Table] RAG vs RAG+Rewrite Summary")
print(df_summary.to_string(index=False))

out_tbl = os.path.join(TABLE_DIR, 'table_rewrite_summary.csv')
df_summary.to_csv(out_tbl, index=False, encoding='utf-8-sig')
print(f"\n[SAVE] table_rewrite_summary.csv")

print(f"\n✅ Notebook 10 complete — Next: 11_rag_model_comparison.ipynb")
print(f"   Then run 04_evaluation_g1_g4.ipynb on rag_rewrite output")

[INFO] LLM model      : gpt-4o-mini
[INFO] Embedding model: text-embedding-3-small
[INFO] Top-k          : 3 (fixed — query rewriting only)
[INFO] Dataset loaded: 300 records
  [LOAD] NIST_AI_RMF          | 11 chunks
  [LOAD] KR_AI_BASIC_ACT      | 17 chunks
  [LOAD] EU_AI_ACT            | 23 chunks
[INFO] All vector stores loaded.
[INFO] Query Rewriting Spot-Check

Regulation : EU_AI_ACT
Original   : What are the requirements for human oversight of high-risk AI credit scoring systems?
Rewritten  : What are the requirements for human oversight of High-Risk Artificial Intelligence credit scoring systems as outlined in Annex III of the EU AI Act (Regulation 2024/1689)?
Tokens used: 218
----------------------------------------------------------------------
Regulation : KR_AI_BASIC_ACT
Original   : What obligations apply to AI business operators providing High-Impact AI for credit screening?
Rewritten  : What obligations apply to artificial intelligence business operators providing High-Im

RAG+Rewrite:  17%|███████████▎                                                        | 50/300 [08:18<43:18, 10.39s/it]

  [Checkpoint  50/300] errors: 0 | avg response: 2818 chars | inf tokens: 102,326 | rew tokens: 13,277


RAG+Rewrite:  33%|██████████████████████▎                                            | 100/300 [19:21<38:00, 11.40s/it]

  [Checkpoint 100/300] errors: 0 | avg response: 2845 chars | inf tokens: 209,044 | rew tokens: 26,407


...

  [Checkpoint 200/300] errors: 0 | avg response: 2768 chars | inf tokens: 403,300 | rew tokens: 53,073


RAG+Rewrite:  83%|███████████████████████████████████████████████████████▊           | 250/300 [46:18<07:49,  9.40s/it]

  [Checkpoint 250/300] errors: 0 | avg response: 2693 chars | inf tokens: 510,745 | rew tokens: 66,466


RAG+Rewrite: 100%|███████████████████████████████████████████████████████████████████| 300/300 [54:33<00:00, 10.91s/it]

  [Checkpoint 300/300] errors: 0 | avg response: 2688 chars | inf tokens: 617,538 | rew tokens: 80,012

[SAVE] responses_rag_rewrite.json
[INFO] records: 300 | errors: 0
[INFO] inference tokens : 617,538
[INFO] rewrite tokens   : 80,012
[INFO] total tokens     : 697,550 | estimated cost: $0.1046
[INFO] Rewriting Quality Analysis

  Avg original question length  : 208 chars
  Avg rewritten question length : 321 chars
  Avg length increase           : +113 chars (+54.2%)

  Rewrite token usage by regulation:
    EU_AI_ACT            | avg: 269 | total: 26,939
    KR_AI_BASIC_ACT      | avg: 267 | total: 26,666
    NIST_AI_RMF          | avg: 264 | total: 26,407

  Sample rewrite pairs:
  Regulation : EU_AI_ACT
  Original   : Article 9(10) of the EU AI Act allows financial institutions' AI risk management to be integrated wi...
  Rewritten  : How can the AI Review Committee utilize the provision in Article 9(10) of the EU AI Act (Regulation ...
--------------------------------------------